<a id="preparation-cost-and-state-windows"></a>

# Preparation costs and complexity profiles

Use a lower bound and feasible programs to bracket optimal preparation cost. Window RCC shows how state-side complexity is distributed across subsystems. Compare candidate programs in reset and cooling models, examine Bell and GHZ subsystems, and see which graph-state measurements give a stronger preparation-cost bound. At the end, export a calculation for a colleague to verify.

Use rcc-refcert 0.4.0 or later. Follow the [source quickstart](README.md#install-from-source) and [notebook setup](README.md#jupyter-notebook), select **Python (rcc-refcert)** as the kernel, then run the cells in order.

You can change the requests below and rerun them. The [workflow guide](docs/COST_AND_WINDOWS.md) describes exact parameters, your own matrices, tensor labels and numerical evidence.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

from rcc_refcert.bundle import verify_analysis_bundle
from rcc_refcert.workflows import (
    analyze_cost,
    analyze_windows,
    export_probe,
    load_workflow_template,
    render_workflow_result,
    save_workflow_bundle,
)

## Bound a complete preparation task

The cyclic-reset template fixes a four-qubit model, a complete target in its original basis, and candidate programs. Each physical action costs one atomic slot, including halt. An exact reference-envelope calculation excludes smaller budgets; executing a candidate establishes a feasible upper endpoint when it reaches the target tolerance.

For the default four-qubit model, varying the trace-distance tolerance gives:

| Tolerance | Cost interval | Interpretation |
|---|---|---|
| `1/10` | `[5,5]` | Five slots are necessary and sufficient in this model. |
| `9/20` | `[4,4]` | The four-slot candidate now meets the target. |
| `11/25` | `[4,5]` | The lower bound and best supplied candidate leave a one-slot gap. |

A candidate can miss the target while the analysis completes successfully. To explore a different physical instance, edit the template's `qubits` and `p` parameters before calling `analyze_cost`; use exact strings such as `"19/20"` for probabilities.

In [ ]:
request = load_workflow_template("cyclic-reset")
request["epsilon_values"] = ["1/10", "9/20", "11/25"]
result = analyze_cost(request)
print(render_workflow_result(result))

<a id="locate-structure-with-subsystem-windows"></a>

<a id="compare-subsystem-entropy-and-correlations"></a>

## Compare subsystem complexity

A **subsystem window** retains a chosen group of qubits and traces out the rest, giving a reduced state whose RCC complexity is evaluated under the same reference and control scale.

The first probe compares paired Bell states with a four-qubit GHZ state. At the default parameters, both are globally pure. With `Gamma=4`, their global RCC complexity is 2 st_R; windows 12, 13 and 34 give `(1,0,1)` for the Bell pairs and `(1/2,1/2,1/2)` st_R for GHZ. The cross window shows whether the same global structure is concentrated inside a pair or shared across the register.

Each window reports state-side complexity `C_R,A = [log2(d_A) - S(rho_A)] / g_R` in R-structons (`st_R`), with `g_R=log2(Gamma)` and `1 st_R = g_R bits`. The entropy deficit supplies the information numerator; the control bandwidth calibrates it to the resource scale. The corresponding full-space effective state is `rho_A ⊗ I_B/d_B`, which has the same deficit relative to the full reference. A bipartition resolves `C_R,AB = C_R,A + C_R,B + I(A:B)/g_R`; the correlation contribution uses the same complexity units. The full finite-accuracy preparation-cost bound additionally uses one-shot corrections and model inputs.

The cell compares the pure-state probe with a common uniform-noise mixture. Changing `mixing` keeps the two global spectra equal but changes the window profiles; `gamma` sets the declared comparison scale. The collection also includes `w2-noise-distribution`, which distributes phase noise differently at matched global entropy. Window entropies are numerical, with input and precision diagnostics retained.


In [ ]:
windows = analyze_windows(export_probe("w1-quantum-organization"))
print(render_workflow_result(windows))

mixed_windows = analyze_windows(
    export_probe("w1-quantum-organization", parameters={"mixing": "1/4"})
)
for profile in mixed_windows["profiles"]:
    values = [round(q["rcc_main_term"], 6) for q in profile["windows"]]
    print(profile["id"], "window C after uniform mixing:", values)

## Follow complexity along a cooling preparation

The cooling probe refreshes two bath qubits and applies reversible compression to concentrate purity on the target qubit. Its windows follow the actual candidate execution, with the action and cumulative cost shown at each step.

At tolerance `1/50`, the gain bound excludes costs below six and the six-action candidate reaches the complete target within tolerance, giving a rigorous `[6,6]` cost interval. The eight-action target program supplies the target itself. The reference scale is `g_R=log2(3)`, derived from this model's action classes.

Inspect refresh steps to see how entropy removal changes the complexity profile and resets target–bath correlation; compression redistributes local and correlation contributions at a fixed global RCC value. The states are diagonal, so the reported correlation is classical. Halt adds one charged slot and leaves the state unchanged. To compare candidate improvements, hold the model, full target and tolerance fixed.


In [ ]:
cooling_request = export_probe("w3-cooling-trajectory")
cooling = analyze_cost(cooling_request)
print(render_workflow_result(cooling))
print(render_workflow_result(cooling["windows"]))

<a id="which-local-measurements-reveal-a-noisy-cluster-state"></a>

## Which measurements rule out cheaper preparations of a noisy cluster state?

The template generates synthetic counts. At its default noise and shot count, the two color settings use the same total shots as the Z-basis control. At trace-distance tolerance `1/20`, each color certifies at least one slot in the model with 32 atomic action classes. Their intersection certifies at least two slots. The Z-basis result below is an information lower bound in bits. The two binary tests share a 99% coverage guarantee; including the complete histograms gives at least 97% joint coverage at the default settings.

In [ ]:
from rcc_refcert.workflows import analyze_study, study_template

cluster = study_template("cluster", noise="1/20", shots=10000)
measured = analyze_study(cluster)
print(render_workflow_result(measured))

## Can fewer cooling cycles reach the same target?

For the default three-cycle target, a shorter program reaches the complete state within trace distance 1/50. The model's lower bound and that program close the interval at [6,6] atomic slots, including halt.

The template generates joint Z-basis counts. These give confidence intervals for the measured-window RCC and the correlation contribution, whose lower endpoint can be zero. The report separately shows the polarization inferred from calibrated counts. For a future ideal readout with forecast ground-state probability 0.98, 39 shots give at least 90% design power. At least 37 ground-state outcomes would certify a probability above 0.8 at 99% confidence. Changing the bath or cycle count can leave the requested power out of reach within the search budget.

In [ ]:
thermal = analyze_study(study_template("cooling", bath="9/10", cycles=3))
print(render_workflow_result(thermal))

## Share a calculation for independent verification

Here you will share the four-qubit cyclic-reset calculation from the first example. Its bundle includes the full request and the original result. Verification reconstructs the model, checks the evidence and reruns the calculation, assigning the recipient's result a new run identifier. The original evidence remains available for comparison.

Each run of the next cell creates a new persistent folder under `generated_results/workflow-notebook/`, prints the full bundle path and verifies it independently. The file remains there after the notebook closes, and earlier bundles are preserved. Share this file with the producer version; the recipient can run `python -m rcc_refcert bundle verify PATH`.

`historical_results_supported` should be `True`. This cyclic-reset bundle is readable with 0.3.0 or later; the [bundle guide](docs/ANALYSIS_BUNDLES.md#complete-cost-and-window-workflows) explains the result fields. To apply the workflows to your own matrices or channels, follow the [complete numerical model example](examples/numerical_cost_workflow.py).


In [ ]:
output_root = Path("generated_results") / "workflow-notebook"
output_root.mkdir(parents=True, exist_ok=True)
output_directory = Path(mkdtemp(prefix="cost-", dir=output_root))
bundle = output_directory / "cost.rccbundle"
save_workflow_bundle(bundle, request, result)
verification = verify_analysis_bundle(bundle)
assert verification["fresh_result"]["run_id"] != result["run_id"]
print("Saved bundle:", bundle.resolve())
print("Fresh verification:", verification["fresh_verification"]["outcome"])
print("Historical results supported:", verification["historical_results_supported"])